# Pipeline Operativo: run_all -> rebalance -> apply

Notebook operativo para ejecutar el flujo principal:
1. Ejecuta `scripts/06_run_all.py`
2. Resume métricas clave del modelo/backtest/recomendación
3. Previsualiza el rebalance (dry-run)
4. Opcionalmente ejecuta `--apply` y muestra el resultado
5. Genera histórico + dashboard diario (`scripts/17` y `scripts/18`)
6. Evalúa y ejecuta auto rebalance solo cuando toca (`scripts/19`)


In [ ]:
from __future__ import annotations
import json
import shlex
import subprocess
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
def infer_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "scripts").exists() and (cwd / "configs").exists():
        return cwd
    if cwd.name == "notebooks" and (cwd.parent / "scripts").exists():
        return cwd.parent
    raise RuntimeError("No se pudo inferir PROJECT_ROOT. Abre el notebook desde la carpeta del repo.")
PROJECT_ROOT = infer_project_root()
PYTHON_BIN = PROJECT_ROOT / ".venv/bin/python"
RUN_ALL_SCRIPT = PROJECT_ROOT / "scripts/06_run_all.py"
REBAL_SCRIPT = PROJECT_ROOT / "scripts/04_rebalance.py"
RECOMMENDATION_PATH = PROJECT_ROOT / "outputs/run_all/recommendation.json"
HISTORY_SCRIPT = PROJECT_ROOT / "scripts/17_execution_history.py"
DASH_SCRIPT = PROJECT_ROOT / "scripts/18_execution_dashboard.py"
AUTO_REBAL_SCRIPT = PROJECT_ROOT / "scripts/19_auto_rebalance_if_due.py"
AUTO_APPLY_ORDERS = False   # IMPORTANT (can be set to True)
AUTO_FORCE = False          # IMPORTANT (force cycle even if cadence is not due)
AUTO_SKIP_RUN_ALL = False   # If True, automation uses existing recommendation
APPLY_ORDERS = False        # IMPORTANT (can be set to True)
MAX_SIGNAL_AGE_BDAYS = 3
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"PYTHON_BIN:   {PYTHON_BIN}")
print(f"APPLY_ORDERS: {APPLY_ORDERS}")
print(f"AUTO_APPLY_ORDERS: {AUTO_APPLY_ORDERS}")
print(f"AUTO_FORCE: {AUTO_FORCE}")
print(f"AUTO_SKIP_RUN_ALL: {AUTO_SKIP_RUN_ALL}")


In [ ]:
def run_cmd(cmd: list[str], cwd: Path = PROJECT_ROOT) -> subprocess.CompletedProcess[str]:
    printable = " ".join(shlex.quote(c) for c in cmd)
    print(f"$ {printable}")
    proc = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed (code={proc.returncode}): {printable}")
    return proc


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)


## 1) Ejecutar run_all

In [ ]:
run_cmd([str(PYTHON_BIN), str(RUN_ALL_SCRIPT)])

## 2) Métricas clave y pesos recomendados

In [ ]:
recommendation = load_json(RECOMMENDATION_PATH)
train_summary = load_json(Path(recommendation["artifacts"]["train_summary_path"]))
predict_live_summary = load_json(Path(recommendation["artifacts"]["predict_live_summary_path"]))

overview = {
    "recommended_strategy": recommendation.get("recommended_strategy"),
    "live_signal_date": recommendation.get("live_signal_date"),
    "backtest_rebalance_date": recommendation.get("rebalance_date"),
    "strategy_score_long_only": recommendation.get("strategy_scores", {}).get("long_only"),
    "strategy_score_market_neutral": recommendation.get("strategy_scores", {}).get("market_neutral"),
    "model_oos_ic_spearman": train_summary.get("oos_ic_spearman"),
    "model_oos_cs_ic_mean": train_summary.get("oos_cs_ic_mean"),
    "model_oos_cs_ic_ir": train_summary.get("oos_cs_ic_ir"),
    "model_oos_top_bottom_tstat": train_summary.get("oos_top_bottom_tstat"),
    "live_assets": predict_live_summary.get("n_live_assets"),
    "live_context_stale_business_days": predict_live_summary.get("market_context_stale_business_days"),
    "live_context_fallback_applied": predict_live_summary.get("market_context_fallback_applied"),
}
display(pd.DataFrame([overview]).T.rename(columns={0: "value"}))

strategy_rows = []
for mode, stats in recommendation.get("strategy_summaries", {}).items():
    strategy_rows.append(
        {
            "mode": mode,
            "weekly_sharpe": stats.get("weekly_sharpe_ratio"),
            "max_drawdown": stats.get("max_drawdown"),
            "annualized_return": stats.get("annualized_return"),
            "annualized_vol": stats.get("annualized_volatility"),
            "average_turnover": stats.get("average_turnover"),
        }
    )
display(pd.DataFrame(strategy_rows).sort_values("weekly_sharpe", ascending=False))

weights_path = Path(recommendation["artifacts"]["recommended_weights_csv"])
weights_df = pd.read_csv(weights_path)
print(f"Weights file: {weights_path}")
print(f"n_positions: {len(weights_df)} | gross_exposure: {weights_df['weight'].sum():.6f}")
display(weights_df.sort_values("weight", ascending=False).head(15))


## 3) Previsualizar rebalance (dry-run)

In [ ]:
dry_run_cmd = [
    str(PYTHON_BIN),
    str(REBAL_SCRIPT),
    "--weights-source",
    "run_all",
    "--recommendation-path",
    str(RECOMMENDATION_PATH),
    "--max-signal-age-business-days",
    str(MAX_SIGNAL_AGE_BDAYS),
]
run_cmd(dry_run_cmd)

latest_orders_path = PROJECT_ROOT / "outputs/execution/rebalance_latest_orders.csv"
latest_summary_path = PROJECT_ROOT / "outputs/execution/rebalance_latest_summary.json"

reb_summary = load_json(latest_summary_path)
orders_df = pd.read_csv(latest_orders_path)

summary_view = {
    "broker": reb_summary.get("broker"),
    "apply": reb_summary.get("apply"),
    "rebalance_date": reb_summary.get("rebalance_date"),
    "n_orders": reb_summary.get("n_orders"),
    "portfolio_turnover_estimate": reb_summary.get("portfolio_turnover_estimate"),
    "order_type": reb_summary.get("order_type"),
    "tif": reb_summary.get("tif"),
    "limit_price_offset_bps": reb_summary.get("limit_price_offset_bps"),
}
display(pd.DataFrame([summary_view]).T.rename(columns={0: "value"}))
display(orders_df.sort_values("abs_weight_change", ascending=False).head(20))


## 4) Ejecutar rebalance con --apply (opcional)

Para evitar ejecuciones accidentales, esta celda solo manda ordenes si `APPLY_ORDERS=True`.

In [ ]:
if not APPLY_ORDERS:
    print("APPLY_ORDERS=False -> no se enviaron ordenes. Cambia el flag a True para ejecutar --apply.")
else:
    apply_cmd = dry_run_cmd + ["--apply"]
    run_cmd(apply_cmd)

    latest_summary_path = PROJECT_ROOT / "outputs/execution/rebalance_latest_summary.json"
    latest_orders_path = PROJECT_ROOT / "outputs/execution/rebalance_latest_orders.csv"
    applied_summary = load_json(latest_summary_path)
    applied_orders = pd.read_csv(latest_orders_path)

    print(json.dumps(applied_summary, indent=2))
    display(applied_orders.head(20))


## 5) Histórico de ejecución + dashboard diario

Genera y muestra:
- `outputs/execution/rebalance_history.csv`
- `outputs/execution/rebalance_history_summary.json`
- `outputs/execution/dashboard_daily.csv`
- `outputs/execution/dashboard_daily.png`


In [ ]:
run_cmd([str(PYTHON_BIN), str(HISTORY_SCRIPT)])
run_cmd([str(PYTHON_BIN), str(DASH_SCRIPT)])

history_summary_path = PROJECT_ROOT / "outputs/execution/rebalance_history_summary.json"
dashboard_daily_path = PROJECT_ROOT / "outputs/execution/dashboard_daily.csv"
dashboard_png_path = PROJECT_ROOT / "outputs/execution/dashboard_daily.png"

history_summary = load_json(history_summary_path)
display(pd.DataFrame([history_summary]).T.rename(columns={0: "value"}))

dashboard_daily = pd.read_csv(dashboard_daily_path)
display(dashboard_daily.tail(10))

if dashboard_png_path.exists():
    display(Image(filename=str(dashboard_png_path)))


## 6) Auto rebalance si toca (cada `rebalance_every_n_days`)

Esta celda evalúa si ya toca rebalance según:
- `configs/config_backtest.yaml` (`rebalance_every_n_days`)
- última fecha de precios limpios
- último rebalance ejecutado (`apply=true`)

Solo ejecuta el ciclo si está vencido (o si `AUTO_FORCE=True`).
Para evitar ejecuciones accidentales, `AUTO_APPLY_ORDERS=False` por defecto.


In [ ]:
auto_cmd = [
    str(PYTHON_BIN),
    str(AUTO_REBAL_SCRIPT),
    "--config-data", str(PROJECT_ROOT / "configs/config_data.yaml"),
    "--config-model", str(PROJECT_ROOT / "configs/config_model.yaml"),
    "--config-backtest", str(PROJECT_ROOT / "configs/config_backtest.yaml"),
    "--config-execution", str(PROJECT_ROOT / "configs/config_execution.yaml"),
    "--max-signal-age-business-days", str(MAX_SIGNAL_AGE_BDAYS),
    "--top-k", "10",
]

if AUTO_SKIP_RUN_ALL:
    auto_cmd.append("--skip-run-all")
if AUTO_FORCE:
    auto_cmd.append("--force")
if AUTO_APPLY_ORDERS:
    auto_cmd.append("--apply")

run_cmd(auto_cmd)

auto_report_path = PROJECT_ROOT / "outputs/execution/auto_rebalance_latest.json"
auto_report = load_json(auto_report_path)
display(pd.DataFrame([auto_report]).T.rename(columns={0: "value"}))
